# Your own Colab copy

Choose File, Save a copy in Drive before working. Run the setup cell once, then work through the starter. Download your notebook and exported ZIP before disconnecting. Colab files can disappear when a runtime ends. Open Grading & notes on the task page after finishing your attempt.

In [ ]:
from pathlib import Path
import hashlib
import os
import urllib.request
import zipfile

work = Path('/content/icaic-admissions') if Path('/content').exists() else Path.cwd() / 'icaic-admissions'
work.mkdir(parents=True, exist_ok=True)
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')
os.environ.setdefault('SDL_AUDIODRIVER', 'dummy')
os.environ.setdefault('PYGAME_HIDE_SUPPORT_PROMPT', '1')
payload = urllib.request.urlopen('https://elteomc.github.io/icaic-practice-hub/packs/admissions.zip', timeout=90).read()
assert hashlib.sha256(payload).hexdigest() == 'c6d19a6386cc54107beaabf4610e25d2707bae02e6cad3e5052cb2487b6d1d6b', 'The task pack changed. Reopen the latest notebook from the hub.'
archive_path = work / 'task.zip'
archive_path.write_bytes(payload)
with zipfile.ZipFile(archive_path) as archive:
    archive.extractall(work)
os.chdir(work / 'admissions')
import sys
sys.path.insert(0, str(Path.cwd()))
print('Task folder:', Path.cwd())


# University Admissions

Run the baseline, grade it once, then improve it. Read Statement.md for the exact task contract. The target column is Hungarian because the original data is retained.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

train = pd.read_csv('train.csv')
validation = pd.read_csv('validation.csv')
test = pd.read_csv('test.csv')
TARGET = 'Felvételi Eredmény'
print('Training rows:', len(train))
print('Admission rate:', train[TARGET].mean())
train.head()

## 1. Establish a development result

The supplied baseline uses school prestige. Choose any improvements using your development split, then refit on all supplied training rows.

In [ ]:
features = ['Középiskola Presztízse']
fit, dev = train_test_split(train, test_size=0.2, stratify=train[TARGET], random_state=2026)
model = LogisticRegression(max_iter=1000, random_state=2026)
model.fit(fit[features], fit[TARGET])
print('Development ROC AUC:', roc_auc_score(dev[TARGET], model.predict_proba(dev[features])[:, 1]))

## 2. Write a submission

This file must include validation and final-test IDs. Keep their labels sealed until you grade the exported ZIP.

In [ ]:
model.fit(train[features], train[TARGET])
query = pd.concat([validation, test], ignore_index=True)
output = pd.DataFrame({'ID': query['ID'], TARGET: model.predict_proba(query[features])[:, 1]})
output.to_csv('predictions.csv', index=False)
print('Saved predictions.csv with', len(output), 'rows')
output.head()

## 3. Your next experiment

Record one hypothesis before changing the baseline. Consider the numeric and categorical columns, fit preprocessing only on your training split, and inspect errors. Use counties.geojson only if it helps a clear hypothesis. Save a copy of predictions.csv before replacing it.

In [ ]:
# Write your next experiment here


## Download for the hub

Run this export cell to download your ZIP. Open Grading & notes on the task page and follow the grading instructions. If you are practicing independently, finish your attempt before checking the final practice score.

In [ ]:
from pathlib import Path
import zipfile
assert Path('predictions.csv').is_file(), 'Save predictions.csv first'
with zipfile.ZipFile('hub-upload.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    archive.write('predictions.csv', 'predictions.csv')
output = 'hub-upload.zip'
try:
    from google.colab import files
    files.download(output)
except ImportError:
    print('Saved ZIP at', Path(output).resolve())
